# Chapter 5 - \*Oh My Gawd\*: It's Full of Stars
`1996` Friedman, Daniel P. & Matthias Felleisen. <i>The Little Schemer</i>. 4e. [MIT Press](https://mitpress.mit.edu/9780262560993/the-little-schemer/).
```{contents}
```

---

```scheme
(define rember*
  (lambda (a l)
    (cond
      ( (null? l) '()                                                                )
      ( (atom? (car l)) (cond
                          ( (eq?                  (car l) a) (rember* a (cdr l))  )
                          ( else (cons            (car l)    (rember* a (cdr l))) )) )
      ( else                     (cons (rember* a (car l))   (rember* a (cdr l)))    ))))

(define insertR*
  (lambda (new old l)
    (cond
      ( (null? l) '()                                                                                                       )
      ( (atom? (car l)) (cond
                          ( (eq?                         (car l) old) (cons old (cons new (insertR* new old (cdr l))))  )
                          ( else (cons                   (car l)                          (insertR* new old (cdr l)))   ))  )
      ( else                     (cons (insertR* new old (car l))                         (insertR* new old (cdr l)))       ))))

(define occur*
  (lambda (a l)
    (cond
      ( (null? l) 0                                                      )
      ( (atom? (car l)) (cond
                          ( (eq? (car l) a) (add1 (occur* a (cdr l))) )
                          ( else                  (occur* a (cdr l))  )) )
      ( else       (o+ (occur* a (car l))         (occur* a (cdr l)))    ))))

(define subst*
  (lambda (new old l)
    (cond
      ( (null? l) '()                                                                                     )
      ( (atom? (car l)) (cond
                          ( (eq? (car l) old) (cons new                      (subst* new old (cdr l))) )
                          ( else              (cons                 (car l)  (subst* new old (cdr l))) )) )
      ( else                                  (cons (subst* new old (car l)) (subst* new old (cdr l)))    ))))

(define insertL*
  (lambda (new old l)
    (cond
      ( (null? l) '()                                                                                                     )
      ( (atom? (car l)) (cond
                          ( (eq?                         (car l) old) (cons new (cons old (insertL* new old (cdr l)))) )
                          ( else (cons                   (car l)                          (insertL* new old (cdr l)))  )) )
      ( else                     (cons (insertL* new old (car l))                         (insertL* new old (cdr l)))     ))))

(define member*
  (lambda (a l)
    (cond
      ( (null? l) #f                                                   )
      ( (atom? (car l)) (or (eq?       (car l) a) (member* a (cdr l))) )
      ( else            (or (member* a (car l))   (member* a (cdr l))) ))))

(define leftmost
  (lambda (l)
    (cond
      ( (atom? (car l)) (car l)  )
      ( else  (leftmost (car l)) ))))


```

---

```scheme
;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;; Chapter 5 - *Oh My Gawd*: It's Full of Stars

; rember-star
;   remove all occurrences of the given atom from a list
;   inputs:  atom, list
;   output:  list
(define rember*
  (lambda (a l)
    (cond
      ( (null? l) '()                                                                )    ; Is the list empty? If so, then return the empty list.
      ( (atom? (car l)) (cond                                                             ; else - Is the S-expression an atom? Is so, then test for equality.
                          ( (eq?                  (car l) a) (rember* a (cdr l))  )       ;            Equal? If so, then remove the occurrence of the atom and recur on cdr.
                          ( else (cons            (car l)    (rember* a (cdr l))) )) )    ;            else -               cons the occurrence of the atom and recur on cdr.
      ( else                     (cons (rember* a (car l))   (rember* a (cdr l)))    )))) ;        else - cons the recurrence on car to the recurrence on cdr
;                                 ^^^^                                           constructor of lists
;                                      ^^^^^^^^^^^^^^^^^^^                       natural recursion, on car l
;                                                            ^^^^^^^^^^^^^^^^^^^ natural recursion, on cdr l

(rember* 'cup '((coffee) cup ((tea) cup) (and (hick)) cup))                ; ((coffee) ((tea)) (and (hick)))
(rember* 'sauce '(((tomato sauce)) ((bean) sauce) (and ((flying)) sauce))) ; (((tomato)) ((bean)) (and ((flying))))
(lat? '(((tomato sauce)) ((bean) sauce) (and ((flying)) sauce)))           ; #f
(atom? (car '(((tomato sauce)) ((bean) sauce) (and ((flying)) sauce))))    ; #f

; insertR*
;   insert an atom to the right of each occurrence of the given atom in a list
;   inputs: new [atom], old [atom], l [list]
;   output: list
(define insertR*
  (lambda (new old l)
    (cond
      ( (null? l) '()                                                                                                       )
      ( (atom? (car l)) (cond
                          ( (eq?                         (car l) old) (cons old (cons new (insertR* new old (cdr l))))  )
                          ( else (cons                   (car l)                          (insertR* new old (cdr l)))   ))  )
      ( else                     (cons (insertR* new old (car l))                         (insertR* new old (cdr l)))       ))))

(insertR* 'roast 'chuck '((how much (wood)) could ((a (wood) chuck)) (((chuck))) (if (a) ((wood chuck))) could chuck wood))
; ((how much (wood)) could ((a (wood) chuck roast)) (((chuck roast))) (if (a) ((wood chuck roast))) could chuck roast wood)

; The First Commandment (final)
;   When recurring on a list of atoms, lat, ask two questions about it: (null? lat) and else.
;   When recurring on a list of S-expressions, l, ask three questions about it: (null? l), (atom? (car l)), and else
;
; All *-functions ask three questions and recur with the car as well as with the cdr, whenever the car is a list.
;   This is because all *-functions work on lists that are either empty, an atom consed onto a list, or a list consed onto a list.
;
; The Fourth Commandment (final)
;   Always change at least one argument while recurring.
;   When recurring on a list of atoms, lat, use (cdr lat).
;   When recurring on a number, n, use (sub1 n).
;   And when recurring on a list of S-expressions, l, use (car l) and (cdr l) if neither (null? l) nor (atom? (car l)) are true.
;   It must be changed to be closer to termination.
;   The changing argument must be tested in the termination condition:
;   when using cdr, test termination with null? and
;   when using sub1, test termination with zero?

; occur*
;   get the number of occurrences of the given atom in the list
;   inputs: a [atom], l [list]
;   output: number
(define occur*
  (lambda (a l)
    (cond
      ( (null? l) 0                                                      )
      ( (atom? (car l)) (cond
                          ( (eq? (car l) a) (add1 (occur* a (cdr l))) )
                          ( else                  (occur* a (cdr l))  )) )
      ( else       (o+ (occur* a (car l))         (occur* a (cdr l)))    ))))

(occur* 'banana '((banana) (split ((((banana ice))) (cream (banana)) sherbert)) (banana) (bread) (banana brandy))) ; 5

; subst*
;   inputs: new [atom], old [atom], l [list]
;   output: list
(define subst*
  (lambda (new old l)
    (cond
      ( (null? l) '()                                                                                     )
      ( (atom? (car l)) (cond
                          ( (eq? (car l) old) (cons new                      (subst* new old (cdr l))) )
                          ( else              (cons                 (car l)  (subst* new old (cdr l))) )) )
      ( else                                  (cons (subst* new old (car l)) (subst* new old (cdr l)))    ))))

(subst* 'orange 'banana '((banana) (split ((((banana ice))) (cream (banana)) sherbert)) (banana) (bread) (banana brandy)))
; ((orange) (split ((((orange ice))) (cream (orange)) sherbert)) (orange) (bread) (orange brandy))

; insertL*
;   insert an atom to the left of each occurrence of the given atom in a list
;   inputs: new [atom], old [atom], l [list]
;   output: list
(define insertL*
  (lambda (new old l)
    (cond
      ( (null? l) '()                                                                                                     )
      ( (atom? (car l)) (cond
                          ( (eq?                         (car l) old) (cons new (cons old (insertL* new old (cdr l)))) )
                          ( else (cons                   (car l)                          (insertL* new old (cdr l)))  )) )
      ( else                     (cons (insertL* new old (car l))                         (insertL* new old (cdr l)))     ))))

(insertL* 'pecker 'chuck '((how much (wood)) could ((a (wood) chuck)) (((chuck))) (if (a) ((wood chuck))) could chuck wood))
; ((how much (wood)) could ((a (wood) pecker chuck)) (((pecker chuck))) (if (a) ((wood pecker chuck))) could pecker chuck wood)

; member*
;   inputs: a [atom], l [list]
;   output: bool
(define member*
  (lambda (a l)
    (cond
      ( (null? l) #f                                                   )
      ( (atom? (car l)) (or (eq?       (car l) a) (member* a (cdr l))) )
      ( else            (or (member* a (car l))   (member* a (cdr l))) ))))

(member* 'chips '((potato) (chips ((with) fish) (chips)))) ; #t
;                           ^^^^^

; leftmost
;   get the leftmost atom in a non-empty list of S-expressions that does not contain the empty list
(define leftmost
  (lambda (l)
    (cond
      ( (atom? (car l)) (car l)  )
      ( else  (leftmost (car l)) ))))
(leftmost '((potato) (chips ((with) fish) (chips))))       ; potato
(leftmost '(((hot) (tuna (and))) cheese))                  ; hot
;(leftmost '(((() four)) 17 (seventeen)))                  ; no answer
;(leftmost '())                                            ; no answer

(and (atom? (car '(mozzarella pizza))) (eq? (car '(mozzarella pizza)) 'pizza))                       ; #f
(and (atom? (car '((mozzarella mushroom) pizza))) (eq? (car '((mozzarella mushroom) pizza)) 'pizza)) ; #f
(and (atom? (car '(pizza (tastes good)))) (eq? (car '(pizza (tastes good))) 'pizza))                 ; #t

; (or ...) asks questions one at a time until it finds on that is true.
;   Then (or ...) stops, making its value true.
;   If it cannot find a true argument, the value of (or ...) is false.
;
; (and ...) asks questions one at a time until it finds on whose value is false.
;   Then (and ...) stops, making its value false.
;   If none of the expressions are false, (and ...) is true.
;
; It's possible that one of the arguments of (and ...) and (or ...) is not considered
;   because (and ...) stops if the first argument has the value #f
;   and      (or ...) stops if the first argument has the value #t
; (cond ...) also has the property of not considering all of its arguments.
;   Because of this property, however, neither (and ...) nor (or ...) can be defined as functions in terms of (cond ...),
;   though both (and ...) and (or ...) can be expressed as abbreviations of (cond ...)-expressions:
;   (and α β) = (cond (α β) (else #f))
;   (or  α β) = (cond (α #t) (else β))

; eqlist
;   determine whether two lists are equal
;
; each argument may be either empty, an atom consed onto a list, or a list consed onto a list
; 1) empty, empty
; 2) empty, atom consed onto a list
; 3) empty, list consed onto a list
; 4) atom consed onto a list, empty
; 5) atom consed onto a list, atom consed onto a list
; 6) atom consed onto a list, list consed onto a list
; 7) list consed onto a list, empty
; 8) list consed onto a list, atom consed onto a list
; 9) list consed onto a list, list consed onto a list
(define eqlist?
  (lambda (l1 l2)
    (cond
      ( (and (null? l1) (null? l2))                              #t ) ; They are both empty lists.
      ( (and (null? l1) (atom? (car l2)))                        #f )
      (      (null? l1)                                          #f )
      ( (and (atom? (car l1)) (null? l2))                        #f )
      ( (and (atom? (car l1)) (atom? (car l2)))                       ; They are both atoms.
        (and (eqan? (car l1) (car l2)) (eqlist? (cdr l1) (cdr l2))) ) ;   Recur.
      (      (atom? (car l1))                                    #f )
      (      (null? l2)                                          #f )
      (      (atom? (car l2))                                    #f )
      ( else (and (eqlist? (car l1) (car l2)) (eqlist? (cdr l1) (cdr l2))) ))))

(eqlist? '(strawberry ice cream) '(strawberry ice cream))                   ; #t
(eqlist? '(strawberry ice cream) '(strawberry cream ice))                   ; #f
(eqlist? '(banana ((split))) '((banana) (split)))                           ; #f
(eqlist? '(beef ((sausage)) (and (soda))) '(beef ((salami)) (and (soda))))  ; #f
(eqlist? '(beef ((sausage)) (and (soda))) '(beef ((sausage)) (and (soda)))) ; #t
```

---